# Optional dynamic preview: follow the state transitions
# 선택 동적 미리보기: 상태전이 따라가기

Predict 30 minutes of A/B/C concentrations under a prescribed temperature plan. The state is the three current concentrations; controls are temperature, residence time, and feed. Each interval is 0.2 min with constant controls. A transition MLP predicts concentration increments, which are added to the current state.

주어진 온도 계획에서 30분 동안의 A/B/C 농도를 예측합니다. 상태는 현재 농도 세 개이고 입력은 온도·체류시간·유입 농도입니다. 각 0.2분 구간에서는 입력이 일정합니다. 상태전이 MLP가 예측한 농도 변화량을 현재 상태에 더해 다음 상태를 만듭니다.

## Setup / 실행 준비

Extract the complete lab ZIP and run this notebook in that folder, with the scripts and supplied models beside it. Install requirements.txt into this notebook’s Python environment. The cells save new results under guided-results/.

실습 ZIP 전체를 풀고 스크립트·제공 모델과 같은 폴더에서 notebook을 실행합니다. Notebook의 Python 환경에 requirements.txt를 설치합니다. 새 결과는 guided-results/에 저장됩니다.

```text
python -m pip install -r requirements.txt
```

## 1. Compare a rate with a finite-time change / 1. 변화율과 유한 시간 변화량 비교하기

Start from [1.2,0,0] mol/L at 330 K and tau=5 min. The A rate is (1.2−1.2)/5−0.2×1.2=−0.24 mol/(L min); B gains 0.24 and C initially has rate zero. Euler multiplies these initial rates by 0.2 min. The reference integrates rates as concentrations change within the interval.

330 K, tau=5분에서 [1.2,0,0] mol/L로 시작합니다. A 변화율은 (1.2−1.2)/5−0.2×1.2=−0.24 mol/(L min)이고 B는 0.24만큼 늘며 C의 초기 변화율은 0입니다. Euler는 이 초기 변화율에 0.2분을 곱합니다. 기준 계산은 구간 중 농도가 변하면서 변화율도 달라지는 과정을 반영합니다.

In [1]:
from pathlib import Path
import numpy as np
import week02_lab as cstr
initial = np.array([[1.2, 0., 0.]])
control = np.array([[330., 5., 1.2]])
rate = cstr.concentration_rate(initial, control)
euler = initial+.2*rate
exact = cstr.reference_step(initial, control)
print("initial rate:", rate[0], "Euler:", euler[0], "reference:", exact[0])
np.testing.assert_allclose(euler, [[1.152, .048, 0]])
np.testing.assert_allclose(exact[0,0], .6+.6*np.exp(-.4*.2))
np.testing.assert_allclose(exact.sum(1), [1.2])

initial rate: [-0.24  0.24  0.  ] Euler: [1.152 0.048 0.   ] reference: [1.15386981e+00 4.56720112e-02 4.58180965e-04]


The reference A concentration is about 1.153870 mol/L. Its equilibrium is 0.6 and its deviation decays by exp(−0.4×0.2). C becomes positive during the interval because B is produced before the interval ends. A rate has units per minute; an increment has units mol/L.

기준 A 농도는 약 1.153870 mol/L입니다. 평형값 0.6에서의 편차가 exp(−0.4×0.2)배로 줄어듭니다. 구간이 끝나기 전에 B가 생성되므로 C도 양수가 됩니다. 변화율은 분당 값이고 변화량은 mol/L입니다.

## 2. Inspect the first three transitions / 2. 처음 세 상태전이 확인하기

Three input intervals require four state rows: the initial state and three successors. One-step evaluation repeatedly supplies the reference current state. Rollout supplies its own previous prediction after initialization. Keeping the weights and controls identical isolates this feedback difference.

입력 구간 세 개에는 초기 상태를 포함해 상태 네 행이 필요합니다. One-step에는 매번 기준 현재 상태를 넣습니다. Rollout에는 초기화 이후 직전 예측을 넣습니다. 가중치와 입력 계획을 같게 유지하면 이 되먹임 차이를 비교할 수 있습니다.

In [2]:
model, scaling = cstr.load_model(Path("."), "dynamic-model")
controls = np.tile(control[0], (3,1))
truth = cstr.simulate_trajectory(controls, initial[0])
one = cstr.dynamic_step(model, scaling, truth[:-1], controls)
roll = cstr.rollout(model, scaling, controls, initial[0])
for k in range(3):
    print("step", k+1, "reference input", truth[k], "rollout input", roll[k])
    print("next B: reference, one-step, rollout", truth[k+1,1], one[k,1], roll[k+1,1])
assert truth.shape == roll.shape == (4,3)
np.testing.assert_allclose(one[0], roll[1], atol=1e-7)

step 1 reference input [1.2 0.  0. ] rollout input [1.2 0.  0. ]
next B: reference, one-step, rollout 0.045672011203436 0.04478118645110628 0.04478118645110628
step 2 reference input [1.15386981e+00 4.56720112e-02 4.58180965e-04] rollout input [ 1.15574537e+00  4.47811865e-02 -4.59112043e-04]
next B: reference, one-step, rollout 0.08696380261427239 0.0862101054626537 0.08540902919612187
step 3 reference input [1.11128627 0.0869638  0.00174992] rollout input [1.11456026e+00 8.54090292e-02 1.56370000e-04]
next B: reference, one-step, rollout 0.12426273584915354 0.1239053475039215 0.12250736101052973


The first predicted step agrees within floating-point tolerance because the initial states match; batch and single-row matrix operations can round differently. Later predictions can differ because the input states differ even with identical weights. Do not replace roll[k] with truth[k] while claiming to measure rollout.

첫 예측은 두 방식의 초기 상태가 같아 부동소수점 허용오차 안에서 일치합니다. Batch와 한 행의 행렬 연산은 반올림이 조금 다를 수 있습니다. 이후에는 입력 상태가 달라져 같은 가중치로도 다른 예측이 나옵니다. Rollout을 측정할 때 roll[k]를 truth[k]로 교체하면 안 됩니다.

## 3. Evaluate complete held-out trajectories / 3. 독립 궤적 전체 평가하기

Split entire trajectories, rather than adjacent rows from the same trajectory, to avoid sharing a process history between training and test. There are 60/15/20 trajectories with 150 intervals each. Exclude the supplied initial state from error averages: it was not predicted.

인접 행을 나누는 대신 궤적 전체를 train·validation·test로 나눠 같은 공정 이력이 양쪽에 섞이지 않게 합니다. 궤적은 60/15/20개이고 각 150구간입니다. 주어진 초기 상태는 예측값이 아니므로 오차 평균에서 제외합니다.

In [3]:
trajectories = cstr.generate_trajectories(seed=42)
inputs, increments = cstr.transition_data(trajectories["test"])
one = cstr.dynamic_step(model,scaling,inputs[:,:3],inputs[:,3:])
predictions = [cstr.rollout(model,scaling,t["controls"],t["states"][0]) for t in trajectories["test"]]
truth_all = np.concatenate([t["states"][1:] for t in trajectories["test"]])
pred_all = np.concatenate([t[1:] for t in predictions])
feed = np.concatenate([t["controls"][:,2] for t in trajectories["test"]])
metrics = {"one_step":cstr.prediction_metrics(inputs[:,:3]+increments,one,inputs[:,-1]),
           "rollout":cstr.prediction_metrics(truth_all,pred_all,feed)}
manual = np.sqrt(np.mean((pred_all-truth_all)**2,axis=0))
np.testing.assert_allclose(manual,metrics["rollout"]["rmse_mol_L"])
assert len(truth_all)==3000
print("one-step and rollout metrics:",metrics)
out = Path("guided-results");out.mkdir(exist_ok=True)
cstr.write_json(out/"dynamic-preview-results.json",metrics)
cstr.dynamic_figures(out,trajectories["test"],predictions,metrics["one_step"]["rmse_mol_L"])

one-step and rollout metrics: {'one_step': {'rmse_mol_L': [0.0011698733687232174, 0.0012881459751205937, 0.0011631600085516736], 'balance_rmse_mol_L': 5.1329873751630126e-05, 'minimum_predicted_concentration_mol_L': -0.001152203213085274}, 'rollout': {'rmse_mol_L': [0.009465025357851294, 0.009629144427021242, 0.010425194376398996], 'balance_rmse_mol_L': 0.002013982987355719, 'minimum_predicted_concentration_mol_L': -0.00516641000457166}}


Read dynamic-rollout.png together with dynamic-errors.png. A small one-step error can feed into later states and accumulate during rollout. The two plots answer different questions: trajectory shape and error versus forecast distance. Also report sum residual and negative concentrations from the metrics. To train a fresh model after this frozen-model comparison, run `python week02_lab.py --help` and use its dynamic mode; keep the evaluation protocols unchanged.

dynamic-rollout.png와 dynamic-errors.png를 함께 읽습니다. 작은 한 단계 오차도 다음 상태에 들어가며 누적될 수 있습니다. 두 그림은 각각 궤적 모양과 예측 거리별 오차를 보여줍니다. 지표에서 농도 합 residual과 음수 농도도 확인하세요. 고정 모델 비교 후 새 모델을 학습하려면 `python week02_lab.py --help`의 dynamic 모드를 사용하되 두 평가 방식을 그대로 유지하세요.

## 4. Change the initial state, keep the future plan / 4. 미래 입력을 고정하고 초기 상태 바꾸기

Use [0.2,0.8,0.2] mol/L instead of pure A. This changes the transient while preserving total concentration. Keep the same 150-interval 330 K plan so that the effect comes from initial state, not a new control sequence.

순수 A 대신 [0.2,0.8,0.2] mol/L를 사용합니다. 농도 합은 같고 과도응답이 바뀝니다. 150구간의 330 K 계획은 그대로 두어 초기 상태의 영향만 비교합니다.

In [4]:
controls = np.tile([330.,5.,1.2], (150,1))
new_initial = np.array([.2,.8,.2])
changed_truth = cstr.simulate_trajectory(controls, new_initial)
changed_pred = cstr.rollout(model, scaling, controls, new_initial)
print("B at 0, 0.2, 30 min:", changed_truth[[0,1,-1],1])
print("changed-state metrics:", cstr.prediction_metrics(changed_truth[1:], changed_pred[1:], controls[:,2]))
np.testing.assert_allclose(changed_truth.sum(1), 1.2, atol=1e-12)

B at 0, 0.2, 30 min: [0.8        0.76178726 0.39995555]
changed-state metrics: {'rmse_mol_L': [0.0066709338814699435, 0.0049788177951477965, 0.004695674747638106], 'balance_rmse_mol_L': 0.0006589654977454142, 'minimum_predicted_concentration_mol_L': 0.19912686245379493}


The reference initially has B above its 0.4 mol/L equilibrium, so B falls at first. Compare this with the original B=0 transient. Report states, controls, units, split rules, both protocols, and the changed-state result. RNN/LSTM comparisons are optional after this transition-model baseline.

기준 B의 초기 농도는 평형값 0.4 mol/L보다 높아 처음에는 감소합니다. B=0에서 시작한 경우와 비교하세요. 상태·입력·단위·분할 방식·두 평가 방식·초기 상태 변경 결과를 보고합니다. RNN/LSTM 비교는 이 상태전이 모델 이후의 선택 과제입니다.

## Report / 보고서

Explain a hand prediction, the corresponding computed intermediate values, and the final interpretation. Change one setting and compare your predicted effect with the observed result. Keep training, validation/calibration, and held-out evaluation roles separate.

손으로 예측한 값, 대응하는 중간 계산값, 최종 해석을 연결해 설명합니다. 설정 하나를 바꾸고 예상한 효과와 실제 결과를 비교합니다. Train·validation/calibration·독립 평가 데이터의 역할을 구분합니다.